# Run the full ETL pipeline (Option 1)
`public` (source) -> `staging` -> `transformed` -> `clean`

**Before you start:** put `rules.sql`, `extract.sql`, `transform.sql`, `load.sql` and `schema.sql` in `SQL_DIR`, and the five CSV files in `DATA_DIR`.
Run the cells top to bottom. Step 2 is destructive for `public` (it reloads it from the CSVs), so it is guarded by `CONFIRM_RESET`.


In [1]:
from pathlib import Path
from io import StringIO
import pandas as pd
from sqlalchemy import text
from connectdb import get_engine

engine = get_engine()

SQL_DIR  = Path("/home/joseph/Projects/Retail-Analytics-Data/etl")    # folder with rules.sql, extract.sql, transform.sql, load.sql, schema.sql
DATA_DIR = Path("/home/joseph/Projects/Retail-Analytics-Data/data")   # folder with fact_sales.csv, dim_*.csv

CONFIRM_RESET   = True   # set True to reload public from the CSVs (step 2)
LOAD_RAW_SALES  = True    # optional: load original sale categories, only used for the audit table

def run_file(path):
    raw = engine.raw_connection()
    try:
        raw.cursor().execute(Path(path).read_text())
        raw.commit()
    finally:
        raw.close()

def query(sql):
    return pd.read_sql(text(sql), engine)
print("connected:", query("select current_database() as db").iloc[0, 0])

connected: postgres


## Step 2: reset `public` from the CSVs
This removes the earlier notebook edits (category update, customer deletes, `customer_dedup_map`, `fact_sales_clean`) and the old pipeline schemas.

In [2]:
if not CONFIRM_RESET:
    raise SystemExit("Set CONFIRM_RESET = True to reload public from the CSVs.")

raw = engine.raw_connection()
cur = raw.cursor()
cur.execute("""
    DROP SCHEMA IF EXISTS clean, transformed, staging, etl CASCADE;
    DROP VIEW  IF EXISTS public.fact_sales_clean;
    DROP TABLE IF EXISTS public.customer_dedup_map, public.fact_sales, public.dim_store,
                         public.dim_customer, public.dim_product, public.dim_date CASCADE;
""")
raw.commit()
cur.execute("SET search_path TO public;\n" + (DATA_DIR / "schema.sql").read_text())
raw.commit()
for table in ["dim_date", "dim_product", "dim_customer", "dim_store", "fact_sales"]:
    with open(DATA_DIR / f"{table}.csv") as fh:
        cur.copy_expert(f"COPY public.{table} FROM STDIN WITH CSV HEADER", fh)
raw.commit()
raw.close()
query("""select 'fact_sales' t, count(*) n from public.fact_sales union all
         select 'dim_product', count(*) from public.dim_product union all
         select 'dim_customer', count(*) from public.dim_customer""")

,t,n
0,fact_sales,60000
1,dim_product,400
2,dim_customer,2000


## Step 3: install the rules and the three stage functions

In [3]:
for name in ["rules", "extract", "transform", "load"]:
    run_file(SQL_DIR / f"{name}.sql")
    print("installed", name)
query("select count(*) as product_names_mapped from etl.product_category_map")

installed rules
installed extract
installed transform
installed load


,product_names_mapped
0,42


## Step 4: run extract -> transform -> load

In [4]:
raw = engine.raw_connection()
conn = getattr(raw, "driver_connection", None) or raw.connection
cur = raw.cursor()

cur.execute("SELECT etl.extract()"); raw.commit()

if LOAD_RAW_SALES:
    df = pd.read_csv(DATA_DIR / "fact_sales.csv", usecols=["sale_id", "category"])
    buf = StringIO(); df.to_csv(buf, index=False); buf.seek(0)
    cur.copy_expert("COPY staging.raw_sales (sale_id, category) FROM STDIN WITH CSV HEADER", buf)
    raw.commit()

del conn.notices[:]
cur.execute("SELECT etl.transform()"); raw.commit()
cur.execute("SELECT etl.load()");      raw.commit()

for n in conn.notices:
    if "Issue" in n:
        print(n.strip())
raw.close()
print("pipeline finished")

NOTICE:  Issue 3: source month_name mismatches = 667
NOTICE:  Issue 5: source products with unit_cost > list_price = 313
NOTICE:  Issue 8: sale category equals product category on 27.5 percent of rows
NOTICE:  Issue 7: stored revenue differs from the standard formula on 99.91 percent of rows
pipeline finished


## Checking and Cleaning dim_product by refering the category column from clean.dim_product

In [5]:
query("""select * from dim_product""")

,product_key,product_name,category,subcategory,unit_cost,list_price
0,1172481,"4K OLED Smart TV 55""",Grocery,Budget,52.77,61.49
1,208984,Wireless Earbuds Pro,Grocery,Budget,32.01,26.99
2,1443361,Shower Curtain Liner,Home,Budget,103.10,132.99
3,680417,Cotton Percale Sheet Set Queen,Home,Premium,98.74,5.00
4,1415258,Smartwatch Series 5,Electronics,Budget,147.27,56.99
...,...,...,...,...,...,...
395,1404443,Smart Plug 4-Pack,Apparel,Budget,128.95,125.71
396,376588,Portable Charger 20000mAh,Electronics,Standard,46.97,40.00
397,1014087,"Tablet 10.5"" WiFi",Grocery,Premium,50.23,62.80
398,1608315,"Cast Iron Skillet 12""",Home,Standard,148.27,33.99


In [6]:
query("""select * from clean.dim_product""")

,product_key,product_name,category,product_group,subcategory,unit_cost,list_price,cost_gt_price,margin_pct,pricing_rule_id
0,1172481,"4K OLED Smart TV 55""",Grocery,Electronics,Budget,100.22,160.35,False,37.5,15
1,208984,Wireless Earbuds Pro,Grocery,Electronics,Budget,93.29,149.26,False,37.5,35
2,1443361,Shower Curtain Liner,Home,Home,Budget,114.26,182.82,False,37.5,22
3,680417,Cotton Percale Sheet Set Queen,Home,Home,Premium,87.31,139.70,False,37.5,26
4,1415258,Smartwatch Series 5,Electronics,Electronics,Budget,92.70,148.32,False,37.5,23
...,...,...,...,...,...,...,...,...,...,...
395,1404443,Smart Plug 4-Pack,Apparel,Electronics,Budget,91.01,145.62,False,37.5,19
396,376588,Portable Charger 20000mAh,Electronics,Electronics,Standard,108.94,174.30,False,37.5,32
397,1014087,"Tablet 10.5"" WiFi",Grocery,Electronics,Premium,151.79,242.86,False,37.5,9
398,1608315,"Cast Iron Skillet 12""",Home,Home,Standard,130.71,209.14,False,37.5,25


In [7]:
with engine.begin() as connection:
    connection.execute(text("""alter table dim_product drop column category"""))

query("""select * from dim_product""")

,product_key,product_name,subcategory,unit_cost,list_price
0,1172481,"4K OLED Smart TV 55""",Budget,52.77,61.49
1,208984,Wireless Earbuds Pro,Budget,32.01,26.99
2,1443361,Shower Curtain Liner,Budget,103.10,132.99
3,680417,Cotton Percale Sheet Set Queen,Premium,98.74,5.00
4,1415258,Smartwatch Series 5,Budget,147.27,56.99
...,...,...,...,...,...
395,1404443,Smart Plug 4-Pack,Budget,128.95,125.71
396,376588,Portable Charger 20000mAh,Standard,46.97,40.00
397,1014087,"Tablet 10.5"" WiFi",Premium,50.23,62.80
398,1608315,"Cast Iron Skillet 12""",Standard,148.27,33.99


In [8]:
with engine.begin() as connection:
    connection.execute(text("""alter table dim_product add column category text"""))

In [9]:
query("""select * from dim_product""")

,product_key,product_name,subcategory,unit_cost,list_price,category
0,1172481,"4K OLED Smart TV 55""",Budget,52.77,61.49,None
1,208984,Wireless Earbuds Pro,Budget,32.01,26.99,None
2,1443361,Shower Curtain Liner,Budget,103.10,132.99,None
3,680417,Cotton Percale Sheet Set Queen,Premium,98.74,5.00,None
4,1415258,Smartwatch Series 5,Budget,147.27,56.99,None
...,...,...,...,...,...,...
395,1404443,Smart Plug 4-Pack,Budget,128.95,125.71,None
396,376588,Portable Charger 20000mAh,Standard,46.97,40.00,None
397,1014087,"Tablet 10.5"" WiFi",Premium,50.23,62.80,None
398,1608315,"Cast Iron Skillet 12""",Standard,148.27,33.99,None


In [10]:
with engine.begin() as connection:
    connection.execute(text("""UPDATE dim_product AS t
SET category = s.product_group
FROM clean.dim_product AS s
WHERE t.product_key= s.product_key;"""))

In [11]:
query("""select * from dim_product""")

,product_key,product_name,subcategory,unit_cost,list_price,category
0,1817693,Air Purifier HEPA,Standard,53.10,5.00,Home
1,1172481,"4K OLED Smart TV 55""",Budget,52.77,61.49,Electronics
2,208984,Wireless Earbuds Pro,Budget,32.01,26.99,Electronics
3,1443361,Shower Curtain Liner,Budget,103.10,132.99,Home
4,680417,Cotton Percale Sheet Set Queen,Premium,98.74,5.00,Home
...,...,...,...,...,...,...
395,1404443,Smart Plug 4-Pack,Budget,128.95,125.71,Electronics
396,376588,Portable Charger 20000mAh,Standard,46.97,40.00,Electronics
397,1014087,"Tablet 10.5"" WiFi",Premium,50.23,62.80,Electronics
398,1608315,"Cast Iron Skillet 12""",Standard,148.27,33.99,Home


## Sincronize updated category column from dim_product into fact_sales.

In [12]:
with engine.begin() as connection:
    connection.execute(text("""UPDATE fact_sales AS fs
SET category = dp.category
FROM dim_product AS dp
WHERE fs.product_key= dp.product_key;"""))

In [14]:
query("""select * from fact_sales""")

,sale_id,date_key,product_key,customer_key,store_key,sold_at,quantity,unit_price,discount_pct,revenue,category
0,632358,1590145,1654389,198445,172884,2024-01-31 23:59:58.518775,2,75.95,0.35,14.98,Home
1,168456,1590145,141352,947644,740545,2024-01-31 23:59:58.134398,1,35.00,0.35,78.00,Home
2,1713987,1763467,770439,1350108,1154336,2024-01-31 23:59:58.076511,1,234.00,0.35,49.91,Home
3,1881479,108350,1304912,1684002,1069986,2024-01-31 23:59:58.885245,1,60.00,0.35,89.30,Home
4,888678,496385,30337,1290362,126408,2024-01-31 23:59:58.483213,1,28.99,0.35,23.61,Home
...,...,...,...,...,...,...,...,...,...,...,...
59995,768576,245370,1852392,1101440,1006546,2025-12-31 23:50:07.690869,1,56.95,0.35,16.93,Home
59996,1322241,307227,614341,1110714,810646,2025-12-31 23:50:15.761791,2,12.00,0.35,156.76,Electronics
59997,481629,769874,1092772,444571,376185,2025-12-31 23:53:12.894208,4,35.99,0.35,28.46,Electronics
59998,435591,1948498,1670378,351876,866773,2025-12-31 23:54:00.847342,1,8.99,0.35,54.00,Electronics


## Adding Brand to Product Name in Dim_Product as the pivot table to handle duplicated data.

In [15]:
query("""select distinct product_name from dim_product""")

,product_name
0,E-Reader Paperwhite
1,"27"" Monitor 144Hz"
2,Silicone Baking Mat Set
3,Cotton Percale Sheet Set Queen
4,Portable SSD 1TB
5,Robot Vacuum Pro
6,Bamboo Cutting Board Set
7,Dash Cam Front & Rear
8,Portable Charger 20000mAh
9,Electric Toothbrush Smart


In [16]:
query("""SELECT product_name, COUNT(*) AS occurrences
FROM dim_product
GROUP BY product_name
ORDER BY occurrences DESC, product_name;""")

,product_name,occurrences
0,Wireless Earbuds Pro,31
1,Robot Vacuum Pro,25
2,Ergonomic Gaming Mouse,24
3,Smart Plug 4-Pack,23
4,"4K OLED Smart TV 55""",18
5,Smart LED Desk Lamp,18
6,Portable Charger 20000mAh,16
7,Dash Cam Front & Rear,15
8,Webcam 1080p,14
9,Cotton Percale Sheet Set Queen,13


In [18]:
with engine.begin() as connection:
    connection.execute(text("""UPDATE dim_product d
SET product_name = (ARRAY[
        'Sony','Samsung','LG','Philips','Apple','Xiaomi','Bose','JBL','Anker','Logitech',
        'Dell','ASUS','Lenovo','Huawei','Panasonic','Sharp','Toshiba','Hisense','TCL','Dyson',
        'Bosch','Siemens','Whirlpool','Electrolux','Tefal','Cuisinart','OXO','Ikea','Belkin','Baseus',
        'TP-Link','Netgear','Razer','Corsair','Garmin','Sennheiser','Haier','Midea','Ninja','Shark'
    ])[n.rn] || ' ' || d.product_name
FROM (
    SELECT product_key,
           ROW_NUMBER() OVER (PARTITION BY product_name ORDER BY product_key) AS rn
    FROM dim_product
) n
WHERE d.product_key = n.product_key;"""))

In [20]:
query("""select * from dim_product""")

,product_key,product_name,subcategory,unit_cost,list_price,category
0,85176,"Sony 27"" Monitor 144Hz",Premium,49.28,62.49,Electronics
1,454946,"LG 27"" Monitor 144Hz",Standard,194.88,5.00,Electronics
2,596204,"Apple 27"" Monitor 144Hz",Standard,76.44,43.99,Electronics
3,1051449,"Xiaomi 27"" Monitor 144Hz",Standard,135.63,27.99,Electronics
4,1097532,"Bose 27"" Monitor 144Hz",Premium,67.22,24.99,Electronics
...,...,...,...,...,...,...
395,824107,Apple Portable SSD 1TB,Premium,248.75,146.49,Electronics
396,1064023,Xiaomi Portable SSD 1TB,Premium,94.20,34.99,Electronics
397,1199323,Apple Pour-Over Coffee Set,Budget,84.24,7.99,Home
398,1387822,Xiaomi Pour-Over Coffee Set,Premium,33.45,25.99,Home


In [21]:
query("""select product_name, COUNT(*) AS occurrences
FROM dim_product
GROUP BY product_name
ORDER BY occurrences DESC, product_name;""")

,product_name,occurrences
0,"Anker 27"" Monitor 144Hz",1
1,"Anker 4K OLED Smart TV 55""",1
2,Anker Cotton Percale Sheet Set Queen,1
3,Anker Dash Cam Front & Rear,1
4,Anker E-Reader Paperwhite,1
...,...,...
395,Xiaomi USB-C Laptop Stand,1
396,Xiaomi Webcam 1080p,1
397,Xiaomi WiFi 6 Router,1
398,Xiaomi Wireless Earbuds Pro,1


In [30]:
query("""select product_name
FROM dim_product
GROUP BY product_name
HAVING COUNT(*) > 1""")

,product_name


## Scrape unit_price from online marketplace

In [32]:
query("""select * from dim_product""")

,product_key,product_name,subcategory,unit_cost,list_price,category
0,85176,"Sony 27"" Monitor 144Hz",Premium,49.28,62.49,Electronics
1,454946,"LG 27"" Monitor 144Hz",Standard,194.88,5.00,Electronics
2,596204,"Apple 27"" Monitor 144Hz",Standard,76.44,43.99,Electronics
3,1051449,"Xiaomi 27"" Monitor 144Hz",Standard,135.63,27.99,Electronics
4,1097532,"Bose 27"" Monitor 144Hz",Premium,67.22,24.99,Electronics
...,...,...,...,...,...,...
395,824107,Apple Portable SSD 1TB,Premium,248.75,146.49,Electronics
396,1064023,Xiaomi Portable SSD 1TB,Premium,94.20,34.99,Electronics
397,1199323,Apple Pour-Over Coffee Set,Budget,84.24,7.99,Home
398,1387822,Xiaomi Pour-Over Coffee Set,Premium,33.45,25.99,Home


In [36]:
with engine.connect() as connection:
    result = connection.execute(text("SELECT DISTINCT product_name FROM dim_product"))
    names = [row[0] for row in result]

In [38]:
print(names)

['Xiaomi Silicone Baking Mat Set', 'Logitech Ergonomic Gaming Mouse', 'Huawei Ergonomic Gaming Mouse', 'Sony Aromatherapy Diffuser', 'LG Electric Toothbrush Smart', 'Sharp Portable Charger 20000mAh', 'LG Ergonomic Gaming Mouse', 'Xiaomi Dash Cam Front & Rear', 'LG WiFi 6 Router', 'Sony Webcam 1080p', 'Philips Portable SSD 1TB', 'Dell Wireless Earbuds Pro', 'LG Wireless Earbuds Pro', 'LG Air Purifier HEPA', 'ASUS Ergonomic Gaming Mouse', 'Xiaomi Portable Charger 20000mAh', 'Samsung Ergonomic Gaming Mouse', 'Xiaomi 27" Monitor 144Hz', 'Apple Cordless Vacuum', 'Xiaomi Wireless Noise-Cancelling Headphones', 'JBL Memory Foam Pillow', 'JBL Wireless Earbuds Pro', 'LG Smart Home Hub', 'Apple Mechanical Keyboard', 'Huawei Portable Charger 20000mAh', 'Xiaomi Electric Toothbrush Smart', 'Logitech 4K OLED Smart TV 55"', 'LG Tablet 10.5" WiFi', 'Sharp Ergonomic Gaming Mouse', 'LG Action Camera 4K', 'LG Bamboo Cutting Board Set', 'Sony Wireless Earbuds Pro', 'Huawei Robot Vacuum Pro', 'Ikea Wireless

In [ ]:
open("products.txt", "w").write("\n".join(names))

10868

In [41]:
data_scraped = pd.read_csv(DATA_DIR / "scraped_prices.csv", usecols=["product_name", "price"])
data_scraped.to_sql("scraped_prices", engine, if_exists="replace", index=False)

400

In [43]:
query("""select * from scraped_prices""")

,product_name,price
0,Xiaomi Silicone Baking Mat Set,NaN
1,Logitech Ergonomic Gaming Mouse,34.99
2,Huawei Ergonomic Gaming Mouse,NaN
3,Sony Aromatherapy Diffuser,NaN
4,LG Electric Toothbrush Smart,NaN
...,...,...
395,Lenovo Wireless Earbuds Pro,24.99
396,"JBL Tablet 10.5"" WiFi",NaN
397,Tefal Robot Vacuum Pro,NaN
398,Anker Ergonomic Gaming Mouse,16.01


In [ ]:
#listing all product names in dim_product and saving to products.txt
names = query("SELECT DISTINCT product_name FROM dim_product")['product_name'].tolist()
open("products.txt", "w").write("\n".join(names))

10868

In [50]:
brand_data = pd.read_csv(DATA_DIR / "brand_prices.csv", usecols=all)
brand_data.to_sql("brand_data", engine, if_exists="replace", index=False)

529

In [51]:
query("""select * from brand_data""")

,base_name,brand,ebay_matches,n_used,price,low,high
0,Silicone Baking Mat Set,Not Specified,6878,84,14.48,5.49,62.99
1,Silicone Baking Mat Set,Kitchen,214,60,16.68,5.95,90.01
2,Silicone Baking Mat Set,uxcell,70,31,22.42,14.70,46.73
3,Silicone Baking Mat Set,Silpat,39,24,20.47,7.00,105.98
4,Silicone Baking Mat Set,LÉKUÉ,24,10,27.50,9.00,67.04
...,...,...,...,...,...,...,...
524,Glass Food Storage Containers,Snapware,89,60,45.00,5.50,119.00
525,Glass Food Storage Containers,Lock & Lock,81,30,34.99,11.54,98.00
526,Glass Food Storage Containers,Kilner,75,11,28.87,18.95,64.34
527,Glass Food Storage Containers,Bormioli Rocco,55,8,41.76,19.95,67.93


In [52]:
query("""select distinct base_name as product_name from brand_data""")

,product_name
0,E-Reader Paperwhite
1,"27"" Monitor 144Hz"
2,Cotton Percale Sheet Set Queen
3,Silicone Baking Mat Set
4,Robot Vacuum Pro
5,Portable SSD 1TB
6,Bamboo Cutting Board Set
7,Dash Cam Front & Rear
8,Portable Charger 20000mAh
9,Electric Toothbrush Smart


In [56]:
query("""select distinct brand from brand_data""")

,brand
0,WeighMax
1,Centon
2,Pure
3,Steam Fast
4,Aeotec
...,...
382,Mikasa
383,Charmast
384,Eufy
385,Reliable


In [65]:
with engine.begin() as connection:
    connection.execute(text("""
        UPDATE brand_data
        SET brand = 'Non Brand'
        WHERE brand IS NULL
           OR LOWER(TRIM(brand)) IN ('', 'n/a', 'na', 'none', 'unknown', 'not specified', 'kitchen')
           OR LOWER(brand) LIKE '%specified%'
    """))

In [66]:
query("""SELECT base_name as Product_Name, brand, price
FROM (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY base_name ORDER BY price) AS rn,
           COUNT(*)     OVER (PARTITION BY base_name)                AS n
    FROM brand_data
) t
WHERE rn = (n + 1) / 2;""")

,product_name,brand,price
0,"27"" Monitor 144Hz",Acer,230.00
1,"4K OLED Smart TV 55""",LG,1404.99
2,Action Camera 4K,AKASO,99.99
3,Air Purifier HEPA,GermGuardian,79.98
4,Aromatherapy Diffuser,Essential,24.99
5,Bamboo Cutting Board Set,VEVOR,24.80
6,"Cast Iron Skillet 12""",Victoria,54.20
7,Ceramic Dinnerware Set 12-Piece,TEMPTATIONS,72.45
8,Cordless Vacuum,Eureka,153.99
9,Cotton Percale Sheet Set Queen,Lacoste,74.99


### Replacing the existed row of products from dim_product with product real price from brand_data. 

In [67]:
query("""select * from dim_product""")

,product_key,product_name,subcategory,unit_cost,list_price,category
0,85176,"Sony 27"" Monitor 144Hz",Premium,49.28,62.49,Electronics
1,454946,"LG 27"" Monitor 144Hz",Standard,194.88,5.00,Electronics
2,596204,"Apple 27"" Monitor 144Hz",Standard,76.44,43.99,Electronics
3,1051449,"Xiaomi 27"" Monitor 144Hz",Standard,135.63,27.99,Electronics
4,1097532,"Bose 27"" Monitor 144Hz",Premium,67.22,24.99,Electronics
...,...,...,...,...,...,...
395,824107,Apple Portable SSD 1TB,Premium,248.75,146.49,Electronics
396,1064023,Xiaomi Portable SSD 1TB,Premium,94.20,34.99,Electronics
397,1199323,Apple Pour-Over Coffee Set,Budget,84.24,7.99,Home
398,1387822,Xiaomi Pour-Over Coffee Set,Premium,33.45,25.99,Home


In [68]:
query("""select * from brand_data""")

,base_name,brand,ebay_matches,n_used,price,low,high
0,Silicone Baking Mat Set,uxcell,70,31,22.42,14.70,46.73
1,Silicone Baking Mat Set,Silpat,39,24,20.47,7.00,105.98
2,Silicone Baking Mat Set,LÉKUÉ,24,10,27.50,9.00,67.04
3,Silicone Baking Mat Set,Nordic Ware,20,2,22.93,10.85,35.00
4,Silicone Baking Mat Set,Mrs. Anderson's Baking,20,12,17.14,12.44,31.26
...,...,...,...,...,...,...,...
524,Instant Pot 6Qt,Non Brand,1003,79,79.00,9.45,239.38
525,Non-Stick Cookware Set 10-Piece,Non Brand,262,93,101.49,27.74,664.98
526,Non-Stick Cookware Set 10-Piece,Non Brand,11,7,83.52,58.00,149.99
527,Glass Food Storage Containers,Non Brand,8966,98,29.99,5.99,129.58


In [69]:
bd = brand_data.sort_values("n_used", ascending=False).drop_duplicates(["base_name", "brand"]).copy()
bd["product_name"] = bd["brand"] + " " + bd["base_name"]
bd[["product_name", "price"]].head()

,product_name,price
65,Not Specified Dash Cam Front & Rear,32.64
439,Gibson Ceramic Dinnerware Set 12-Piece,54.24
438,Not Specified Ceramic Dinnerware Set 12-Piece,68.41
328,VHC Shower Curtain Liner,64.31
331,uxcell Shower Curtain Liner,32.95


In [70]:
dim = pd.read_sql(text("SELECT product_key, product_name FROM dim_product"), engine)
bases = sorted(bd["base_name"].unique(), key=len, reverse=True)
dim["base_name"] = dim["product_name"].map(lambda n: next((b for b in bases if n.endswith(" " + b)), None))
dim["base_name"].isna().sum()

np.int64(0)

In [71]:
dim = dim.merge(bd[["product_name", "price"]], on="product_name", how="left")
print(dim["price"].notna().sum(), "matched")
print(dim["price"].isna().sum(), "without a match")

32 matched
368 without a match


In [72]:
new_name = dim["product_name"].copy()
new_price = dim["price"].copy()
taken = set(dim.loc[dim["price"].notna(), "product_name"])

for base, grp in dim[dim["price"].isna()].groupby("base_name"):
    free = bd[(bd["base_name"] == base) & ~bd["product_name"].isin(taken)].sort_values("ebay_matches", ascending=False)
    for i, (_, row) in zip(grp.index, free.iterrows()):
        new_name[i] = row["product_name"]
        new_price[i] = row["price"]

dim["new_name"] = new_name
dim["new_price"] = new_price

In [73]:
renamed = dim["new_name"] != dim["product_name"]
print(renamed.sum(), "rows replaced by another brand")
print(dim["new_price"].isna().sum(), "rows left untouched")
print(dim["new_name"].duplicated().sum(), "duplicate names (must be 0)")
dim.loc[renamed, ["product_key", "product_name", "new_name", "new_price"]]

276 rows replaced by another brand
92 rows left untouched
0 duplicate names (must be 0)


,product_key,product_name,new_name,new_price
2,596204,"Apple 27"" Monitor 144Hz","Not Specified 27"" Monitor 144Hz",170.99
3,1051449,"Xiaomi 27"" Monitor 144Hz","MSI 27"" Monitor 144Hz",212.50
4,1097532,"Bose 27"" Monitor 144Hz","Acer 27"" Monitor 144Hz",230.00
5,1147594,"JBL 27"" Monitor 144Hz","AOC 27"" Monitor 144Hz",180.16
6,1578170,"Anker 27"" Monitor 144Hz","BenQ 27"" Monitor 144Hz",191.01
...,...,...,...,...
392,905446,JBL Portable Charger 20000mAh,FORCELL Portable Charger 20000mAh,37.99
395,824107,Apple Portable SSD 1TB,Western Digital Portable SSD 1TB,271.24
396,1064023,Xiaomi Portable SSD 1TB,Kingston Portable SSD 1TB,298.98
397,1199323,Apple Pour-Over Coffee Set,Primula Pour-Over Coffee Set,55.63


## Sincronize Updated product unit_price column from dim_product into fact_sales

In [74]:
query("""select * from fact_sales""")

,sale_id,date_key,product_key,customer_key,store_key,sold_at,quantity,unit_price,discount_pct,revenue,category
0,632358,1590145,1654389,198445,172884,2024-01-31 23:59:58.518775,2,75.95,0.35,14.98,Home
1,168456,1590145,141352,947644,740545,2024-01-31 23:59:58.134398,1,35.00,0.35,78.00,Home
2,1713987,1763467,770439,1350108,1154336,2024-01-31 23:59:58.076511,1,234.00,0.35,49.91,Home
3,1881479,108350,1304912,1684002,1069986,2024-01-31 23:59:58.885245,1,60.00,0.35,89.30,Home
4,888678,496385,30337,1290362,126408,2024-01-31 23:59:58.483213,1,28.99,0.35,23.61,Home
...,...,...,...,...,...,...,...,...,...,...,...
59995,768576,245370,1852392,1101440,1006546,2025-12-31 23:50:07.690869,1,56.95,0.35,16.93,Home
59996,1322241,307227,614341,1110714,810646,2025-12-31 23:50:15.761791,2,12.00,0.35,156.76,Electronics
59997,481629,769874,1092772,444571,376185,2025-12-31 23:53:12.894208,4,35.99,0.35,28.46,Electronics
59998,435591,1948498,1670378,351876,866773,2025-12-31 23:54:00.847342,1,8.99,0.35,54.00,Electronics
